# Prepare NIGHTS at 224 px (SPEC §6)

Standalone Colab notebook. Downloads the 100 `ref` + 100 `distort` chunks, keeps only images of
`votes >= 6` triplets, resizes them to 224×224 bicubic and stores one tar per chunk on Drive.
Re-running skips finished chunks. Set `FIRST_RUN = True` for chunk 000 only (O2).

## Config

In [ ]:
import subprocess
import sys

FIRST_RUN = True
DRIVE = '/content/drive/MyDrive/nights224'
WORK = '/content/work'
OUT = '/content/nights224'

CSV_URL = 'https://data.csail.mit.edu/nights/nights.zip'
CSV_MEMBER = 'nights/data.csv'
CSV_SHA256 = '5c8d23041917b34758c4308d7310a20eaaa34ab4ea32e134074950204f840c0a'
CHUNK_URL = 'https://data.csail.mit.edu/nights/nights_chunked/{kind}/{i:03d}.zip'
KINDS = ('ref', 'distort')
SIZE = 224
MIN_VOTES = 6
EXPECTED = {'train': 13900, 'val': 1720, 'test': 1824}

try:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'remotezip'], check=True)
except ImportError:
    pass

## data.csv

In [ ]:
import csv
import hashlib
import io
import os
from collections import defaultdict

from remotezip import RemoteZip

SHARDS = os.path.join(DRIVE, 'shards')
CSV_PATH = os.path.join(DRIVE, 'data.csv')
os.makedirs(SHARDS, exist_ok=True)
os.makedirs(WORK, exist_ok=True)


def sha256(data):
    return hashlib.sha256(data).hexdigest()


if not (os.path.exists(CSV_PATH) and sha256(open(CSV_PATH, 'rb').read()) == CSV_SHA256):
    with RemoteZip(CSV_URL) as z:
        data = z.read(CSV_MEMBER)
    assert sha256(data) == CSV_SHA256, 'data.csv sha256 mismatch'
    with open(CSV_PATH, 'wb') as f:
        f.write(data)

with open(CSV_PATH, newline='') as f:
    ROWS = [r for r in csv.DictReader(f) if int(r['votes']) >= MIN_VOTES]
PATH_COLS = ('ref_path', 'left_path', 'right_path')
KEEP = defaultdict(set)  # (kind, chunk) -> paths, e.g. ('distort', 23) -> {'distort/023/081_1.png', ...}
for r in ROWS:
    for c in PATH_COLS:
        kind, chunk, _ = r[c].split('/')
        KEEP[kind, int(chunk)].add(r[c])
print(f'{len(ROWS)} triplets with votes >= {MIN_VOTES}, {sum(map(len, KEEP.values()))} images to keep')

## Chunk processing

In [ ]:
import shutil
import tarfile
import time
import urllib.request
import zipfile
from collections import Counter
from multiprocessing import Pool

from PIL import Image


def download(url, dst, retries=5):
    for attempt in range(1, retries + 1):
        try:
            with urllib.request.urlopen(url, timeout=60) as r, open(dst, 'wb') as f:
                expected = int(r.headers['Content-Length'])
                shutil.copyfileobj(r, f, 1 << 20)
            if os.path.getsize(dst) == expected:
                return
            raise IOError(f'got {os.path.getsize(dst)} of {expected} bytes')
        except Exception as e:
            print(f'  download attempt {attempt} failed: {e}')
            time.sleep(10 * attempt)
    raise RuntimeError(f'could not download {url}')


def resize(item):
    name, data = item
    im = Image.open(io.BytesIO(data))
    mode = im.mode
    im = im.convert('RGB').resize((SIZE, SIZE), Image.BICUBIC)
    buf = io.BytesIO()
    im.save(buf, format='PNG')
    return name, mode, buf.getvalue()


def shard_path(kind, i):
    return os.path.join(SHARDS, f'{kind}_{i:03d}.tar')


def process_chunk(kind, i, pool):
    dst = shard_path(kind, i)
    if os.path.exists(dst):
        return None
    t0 = time.time()
    zpath = os.path.join(WORK, f'{kind}_{i:03d}.zip')
    download(CHUNK_URL.format(kind=kind, i=i), zpath)
    expected = KEEP[kind, i]
    with zipfile.ZipFile(zpath) as z:
        members = [m for m in z.namelist() if not m.endswith('/')]
        # zip members are '000/329.png'; data.csv paths are 'ref/000/329.png'
        named = {m if m.startswith(kind + '/') else f'{kind}/{m}': m for m in members}
        missing = expected - named.keys()
        assert not missing, f'{kind} {i:03d}: {len(missing)} kept paths not in zip, e.g. {sorted(missing)[:3]}'
        items = [(p, z.read(named[p])) for p in sorted(expected)]
    results = sorted(pool.map(resize, items, chunksize=8))
    tmp = os.path.join(WORK, f'{kind}_{i:03d}.tar')
    with tarfile.open(tmp, 'w') as tar:
        for name, _, data in results:
            info = tarfile.TarInfo(name)
            info.size, info.mtime, info.mode = len(data), 0, 0o644
            tar.addfile(info, io.BytesIO(data))
    part = dst + '.part'
    shutil.copyfile(tmp, part)
    os.replace(part, dst)
    os.remove(zpath)
    os.remove(tmp)
    modes = Counter(m for _, m, _ in results)
    print(f'{kind} {i:03d}: {len(members)} in zip, {len(results)} kept, modes {dict(modes)}, '
          f'{os.path.getsize(dst) / 1e6:.1f} MB, {time.time() - t0:.0f} s')
    return modes

## Run

In [ ]:
chunks = [0] if FIRST_RUN else range(100)
for stale in (f for f in os.listdir(SHARDS) if f.endswith('.part')):
    os.remove(os.path.join(SHARDS, stale))
with Pool(2) as pool:
    for i in chunks:
        for kind in KINDS:
            modes = process_chunk(kind, i, pool)
            if modes and set(modes) != {'RGB'}:
                print(f'  NOTE: non-RGB source images converted to RGB: {dict(modes)}')

done = sorted(f for f in os.listdir(SHARDS) if f.endswith('.tar'))
total = sum(os.path.getsize(os.path.join(SHARDS, f)) for f in done)
print(f'{len(done)}/200 shards on Drive, {total / 1e9:.2f} GB')
if FIRST_RUN:
    sizes = {k: os.path.getsize(shard_path(k, 0)) for k in KINDS}
    print('chunk 000:', {k: f'{v / 1e6:.1f} MB' for k, v in sizes.items()},
          f'→ extrapolated total ×100: {sum(sizes.values()) * 100 / 1e9:.1f} GB')

## O2: Drive → `/content` copy and extract time (chunk 000)

Right after the run, Drive's local cache makes this look faster than it is. For the real number,
*Disconnect and delete runtime*, then run Config, data.csv and this cell.

In [ ]:
t0 = time.time()
tmpdir = os.path.join(WORK, 'o2')
os.makedirs(tmpdir, exist_ok=True)
for kind in KINDS:
    shutil.copyfile(shard_path(kind, 0), os.path.join(tmpdir, f'{kind}_000.tar'))
t_copy = time.time() - t0
for kind in KINDS:
    with tarfile.open(os.path.join(tmpdir, f'{kind}_000.tar')) as tar:
        tar.extractall(os.path.join(tmpdir, 'x'), filter='data')
t_all = time.time() - t0
shutil.rmtree(tmpdir)
print(f'chunk 000: copy {t_copy:.1f} s, copy + extract {t_all:.1f} s → ×100 ≈ {t_all * 100 / 60:.1f} min per session')

## Session setup: Drive → `/content/nights224`

In [ ]:
t0 = time.time()
os.makedirs(OUT, exist_ok=True)
local = os.path.join(WORK, 'shard.tar')
done = sorted(f for f in os.listdir(SHARDS) if f.endswith('.tar'))
for f in done:
    shutil.copyfile(os.path.join(SHARDS, f), local)
    with tarfile.open(local) as tar:
        tar.extractall(OUT, filter='data')
os.remove(local)
shutil.copyfile(CSV_PATH, os.path.join(OUT, 'data.csv'))
print(f'{len(done)} shards extracted to {OUT} in {time.time() - t0:.0f} s')

## Verify

In [ ]:
import random

counts = Counter(r['split'] for r in ROWS)
for split, n in EXPECTED.items():
    print(f'{split:5s} {counts[split]:6d} (expected {n})')
assert dict(counts) == EXPECTED, 'filtered split counts differ from SPEC §6'

paths = sorted({r[c] for r in ROWS for c in PATH_COLS})
present = [p for p in paths if os.path.isfile(os.path.join(OUT, p))]
missing = sorted(set(paths) - set(present))
print(f'{len(paths)} images, {len(missing)} missing' + (f', e.g. {missing[:3]}' if missing else ''))
for p in random.Random(0).sample(present, min(20, len(present))):
    with Image.open(os.path.join(OUT, p)) as im:
        assert im.size == (SIZE, SIZE) and im.mode == 'RGB', (p, im.size, im.mode)
print('sha256 of sorted path list:', sha256('\n'.join(paths).encode()))